In [10]:
import pandas as pd

df = pd.read_csv('../data/raw/telco_churn.csv')
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(0)

In [11]:
df_model=df.drop(columns=['customerID'])
df_model.shape

(7043, 20)

In [12]:
df_model['Churn'] = df_model['Churn'].map({'Yes': 1, 'No': 0})

In [13]:
df_model['Churn'].value_counts()

Churn
0    5174
1    1869
Name: count, dtype: int64

In [14]:
binary_cols = ['gender', 'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']

In [15]:
df_model['gender'] = df_model['gender'].map({'Male': 1, 'Female': 0})

In [16]:
yes_no_cols = ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']

for col in yes_no_cols:
    df_model[col] = df_model[col].map({'Yes': 1, 'No': 0})

In [17]:
df_model[['gender', 'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']].head()

,gender,Partner,Dependents,PhoneService,PaperlessBilling
0,0,1,0,0,1
1,1,0,0,1,0
2,1,0,0,1,1
3,1,0,0,0,0
4,0,0,0,1,1


In [18]:
multi_cat_cols = df_model.select_dtypes(include='str').columns.tolist()
multi_cat_cols

['MultipleLines',
 'InternetService',
 'OnlineSecurity',
 'OnlineBackup',
 'DeviceProtection',
 'TechSupport',
 'StreamingTV',
 'StreamingMovies',
 'Contract',
 'PaymentMethod']

In [19]:
df_model.shape

(7043, 20)

In [20]:
service_dependent_cols = ['MultipleLines', 'OnlineSecurity', 'OnlineBackup',
                            'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']

for col in service_dependent_cols:
    df_model[col] = df_model[col].replace({'No internet service': 'No', 'No phone service': 'No'})

In [21]:
df_model['OnlineSecurity'].unique()

<ArrowStringArray>
['No', 'Yes']
Length: 2, dtype: str

In [22]:
df_model['MultipleLines'].unique()

<ArrowStringArray>
['No', 'Yes']
Length: 2, dtype: str

In [23]:
multi_cat_cols=df_model.select_dtypes(include='str').columns.tolist()
multi_cat_cols

['MultipleLines',
 'InternetService',
 'OnlineSecurity',
 'OnlineBackup',
 'DeviceProtection',
 'TechSupport',
 'StreamingTV',
 'StreamingMovies',
 'Contract',
 'PaymentMethod']

In [24]:
for col in service_dependent_cols:
    df_model[col] = df_model[col].map({'Yes': 1, 'No': 0})

In [25]:
df_model = pd.get_dummies(df_model, columns=['InternetService', 'Contract', 'PaymentMethod'], drop_first=True)

In [26]:
df_model.shape

(7043, 24)

In [27]:
df_model.columns.tolist()

['gender',
 'SeniorCitizen',
 'Partner',
 'Dependents',
 'tenure',
 'PhoneService',
 'MultipleLines',
 'OnlineSecurity',
 'OnlineBackup',
 'DeviceProtection',
 'TechSupport',
 'StreamingTV',
 'StreamingMovies',
 'PaperlessBilling',
 'MonthlyCharges',
 'TotalCharges',
 'Churn',
 'InternetService_Fiber optic',
 'InternetService_No',
 'Contract_One year',
 'Contract_Two year',
 'PaymentMethod_Credit card (automatic)',
 'PaymentMethod_Electronic check',
 'PaymentMethod_Mailed check']

In [28]:
df_model.dtypes

gender                                     int64
SeniorCitizen                              int64
Partner                                    int64
Dependents                                 int64
tenure                                     int64
PhoneService                               int64
MultipleLines                              int64
OnlineSecurity                             int64
OnlineBackup                               int64
DeviceProtection                           int64
TechSupport                                int64
StreamingTV                                int64
StreamingMovies                            int64
PaperlessBilling                           int64
MonthlyCharges                           float64
TotalCharges                             float64
Churn                                      int64
InternetService_Fiber optic                 bool
InternetService_No                          bool
Contract_One year                           bool
Contract_Two year   

In [29]:
x=df_model.drop(columns=['Churn'])
y=df_model['Churn']

In [30]:
x.shape
y.shape

(7043,)

In [31]:
x.shape

(7043, 23)

In [32]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42, stratify=y
)

In [33]:
x_train.shape, x_test.shape, y_train.shape, y_test.shape

((5634, 23), (1409, 23), (5634,), (1409,))

In [34]:
y_train.value_counts(normalize=True)
y_test.value_counts(normalize=True)

Churn
0    0.734564
1    0.265436
Name: proportion, dtype: float64

## Preprocessing Summary (so far)

- Dropped `customerID` (unique identifier, no predictive value).
- Encoded binary Yes/No columns (`Churn`, `gender`, `Partner`, `Dependents`,
  `PhoneService`, `PaperlessBilling`) directly to 0/1.
- Simplified service-dependent columns (`OnlineSecurity`, `TechSupport`, etc.)
  by merging "No internet/phone service" into "No", since it was a structural
  artifact rather than a real answer choice.
- One-hot encoded multi-category columns (`InternetService`, `Contract`,
  `PaymentMethod`) using `drop_first=True` to avoid redundant, perfectly
  correlated columns (multicollinearity).
- Split data into train (80%) and test (20%) sets using `stratify=y` to
  preserve the ~73%/27% churn class ratio in both sets.

In [35]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

In [36]:
x_train_scaled[:5]

array([[ 0.99433624, -0.44177295, -0.96923413, -0.65155653,  0.10237124,
        -3.01309011, -0.85833837, -0.63611103, -0.73554636,  1.37557156,
        -0.64327425,  1.25216312,  1.24796703, -1.20265302, -0.52197565,
        -0.2622572 , -0.88769579, -0.52408075, -0.51278214, -0.56382155,
        -0.52380561,  1.40690298, -0.54384572],
       [ 0.99433624, -0.44177295,  1.03174245,  1.53478624, -0.71174346,
         0.3318852 , -0.85833837,  1.57205259, -0.73554636, -0.72697054,
        -0.64327425, -0.79861799, -0.80130322, -1.20265302,  0.33747781,
        -0.50363479,  1.12651205, -0.52408075, -0.51278214, -0.56382155,
        -0.52380561, -0.71078107,  1.83875676],
       [ 0.99433624, -0.44177295,  1.03174245,  1.53478624, -0.79315493,
        -3.01309011, -0.85833837,  1.57205259,  1.35953361, -0.72697054,
         1.55454692, -0.79861799, -0.80130322, -1.20265302, -0.80901319,
        -0.74988292, -0.88769579, -0.52408075, -0.51278214,  1.77361083,
        -0.52380561, -0.7107

## Feature Scaling

`StandardScaler` was applied to standardize all features (mean=0, std=1).
The scaler was fit only on the training set and then used to transform
both train and test sets, avoiding data leakage from the test set into
the scaling statistics.